# k-NN, Naive Bayes and SVM — Concepts

Slides **45–47**, `Slides_MSc_Data_Science_Week_06_EN.pdf`.

Read the idea, the formula and the small example in that order. The demonstration code is provided.

## Libraries and settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import CategoricalNB
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

## 1. k-nearest neighbours

Find the $k$ nearest training points and predict the majority class. Straight-line distance for two inputs is:
$$d=\sqrt{(x_1-a_1)^2+(x_2-a_2)^2}.$$
$x$ is the new point; $a$ is a training point. If three of five neighbours survived, predicted survival probability is $3/5=0.6$.

Small $k$ is sensitive to individual points; larger $k$ makes smoother predictions but can miss local patterns. Choose $k$ using training CV. Fitting mainly stores the examples; prediction must compare the new point with stored observations.

In [ ]:
new_point = np.array([2, 3])
known_point = np.array([5, 7])
print('Distance:', np.sqrt(np.sum((new_point - known_point) ** 2)))
print('Probability from 3 of 5 neighbours:', 3 / 5)

## 2. Why scaling matters

Inputs such as age and fare have different units and spreads. Without scaling, a large numerical spread can dominate distances.
$$z=\frac{x-\mu}{s}.$$
$\mu$ and $s$ are the training mean and standard deviation. `StandardScaler` performs this transformation. Learn it on training data and use those same values for new rows.

A **pipeline** combines scaler and model. In CV, each training fold fits its own scaler. The exercise supplies this structure. Scaling helps k-NN and SVM; classification trees do not require it.

## 3. Naive Bayes step by step

Bayes' theorem updates a prior probability after seeing inputs:
$$P(y\mid x)=\frac{P(x\mid y)P(y)}{P(x)}.$$
$P(y)$ is the prior class share; $P(x\mid y)$ says how typical the inputs are for that class. Naive Bayes assumes that inputs are independent **given the class**, so its score is:
$$\mathrm{score}(y)=P(y)\prod_j P(x_j\mid y).$$
$\prod$ means multiply. Compare the class scores, then divide each by their sum to get probabilities.

For the golf example on slide 46: score(yes) = $(9/14)(2/9)(3/9)(3/9)(3/9)\approx0.0053$; score(no) = $(5/14)(3/5)(1/5)(4/5)(3/5)\approx0.0206$. Predict no, with normalised probability about 0.795. A zero frequency can make a whole product zero; **smoothing** adds small counts. `CategoricalNB` uses smoothing by default. Conditional independence is an assumption, not a fact about all data.

Naive Bayes is also used with word counts for text classification, but that extension is outside the required exercise.

In [ ]:
yes = (9 / 14) * (2 / 9) * (3 / 9) * (3 / 9) * (3 / 9)
no = (5 / 14) * (3 / 5) * (1 / 5) * (4 / 5) * (3 / 5)
print('Scores:', yes, no)
print('P(no):', no / (yes + no))

## 4. SVM: margin, support vectors, C and kernel

An SVM looks for a separating boundary with a wide **margin**. Points that determine the solution are **support vectors**; they can also be inside the margin or misclassified in a soft-margin SVM.

| Setting | Meaning |
|---|---|
| small `C` | more tolerance for errors; stronger preference for a wide margin |
| large `C` | stronger penalty for errors |
| `kernel='linear'` | straight separating boundary |
| `kernel='rbf'` | can form a curved boundary |

A linear score is $s=w^Tx+b$: multiply each input by its weight, add them, then add an intercept. A simplified training objective is:
$$\frac12\lVert w\rVert^2+C\sum_i\max(0,1-y_i s_i).$$
Labels are −1/+1 here. The first term favours a wide margin; the second penalises points inside the margin or on the wrong side. No optimisation derivation is required. A kernel measures similarity as if inputs had been transformed.

The demonstration below uses two simulated inputs so that the boundary is visible.

In [ ]:
from sklearn.datasets import make_moons
points, labels = make_moons(n_samples=80, noise=0.12, random_state=42)
a, b = np.meshgrid(np.linspace(-1.5, 2.5, 100), np.linspace(-1, 1.5, 100))
grid = np.column_stack([a.ravel(), b.ravel()])
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, kernel in zip(axes, ['linear', 'rbf']):
    svm = make_pipeline(StandardScaler(), SVC(kernel=kernel, C=1))
    svm.fit(points, labels)
    scores = svm.decision_function(grid).reshape(a.shape)
    ax.contour(a, b, scores, levels=[-1, 0, 1], colors=['grey', 'black', 'grey'])
    ax.scatter(points[:, 0], points[:, 1], c=labels, cmap='coolwarm')
    support = points[svm[-1].support_]
    ax.scatter(support[:, 0], support[:, 1], s=80, facecolors='none', edgecolors='black')
    ax.set(title=kernel, xlabel='Input 1', ylabel='Input 2')
plt.tight_layout()
plt.show()

### Jupyter notebook — footer info

Include this information when submitting your notebook.

In [ ]:
import platform
from datetime import datetime
print(platform.platform())
print("Python:", platform.python_version())
print("Date:", datetime.now().isoformat(timespec="seconds"))